# Comparing Models with `FitResults`

This notebook exercises the **save / load / compare** workflow added in the `fit-saving` branch:

1. Fit two competing models (`A` and `B`) on the same data, at three different fit levels.
2. Use `file.compare_models(...)` to rank them by fit-quality metrics from `Project._fit_history` (no disk I/O required).
3. Persist the results with `file.save_fit("comparison.fit.h5")`.
4. Reload from disk with `trspecfit.FitResults.load(...)` and run the same comparison without a live `Project`.
5. Exercise `sbs_aggregation="long"` to inspect per-slice metrics for a Slice-by-Slice fit.

**Three comparison stories**, each isolating one structural choice:

| Section | Model A (`win` expected) | Model B | What it shows |
|---------|--------------------------|---------|----------------|
| `baseline` | `baseA` — full GLP | `baseB` — Gaussian only (`m = 0`) | Shape matters — Lorentzian tails *need* the GLP `m` parameter. |
| `sbs`      | `sbsA` — only `x0` floats per slice | `sbsB` — `x0` *and* `A` float per slice | Parsimony matters — the extra free amplitude doesn't earn its keep when only `x0` truly evolves. |
| `2d`       | `m2dA` — `MonoExpPosIRF` (exp ⊗ Gaussian IRF) | `m2dB` — `MonoExpPos` (sharp turn-on) | Instrument response matters — the IRF's extra parameter is paid for by the smooth onset at `t = 0`. |

Synthetic data is generated inline so the ground truth is visible and tunable. Set `M_TRUE → 0` to neuter the baseline comparison; set `SD_TRUE → 0` to neuter the 2D comparison.

`auto_export: False` is set in `project.yaml` so fit-completion side effects (CSV/PNG drops) stay out of the way; this notebook only writes the artifacts it asks for explicitly.

In [ ]:
import os
import numpy as np
import trspecfit

## 1. Generate Synthetic Data

We build the dataset inline so the ground truth is part of the notebook narrative — change a constant and re-run to see how the comparison metrics react.

The dynamics follow the canonical **kicked-decay** pump-probe pattern:

- `t < 0`: peak sits at `X0_BASE = 8` (ground state).
- `t = 0`: pump kicks the peak by `DELTA_X = 2` (to position 10).
- `t > 0`: peak relaxes back to `X0_BASE` with time constant `TAU = 30`.

The pump pulse has finite duration, so the kick is convolved with a **Gaussian IRF** of width `SD_TRUE = 3`. That convolution is what the `m2dA` model needs to capture and `m2dB` cannot.

- **Peak shape:** GLP with `M_TRUE = 0.85` (strongly Lorentzian).
- **Background:** linear, `m = 0.05`, `b = 1.0`.
- **Noise:** Gaussian, σ = 2% of clean-data max.
- **Shape:** `(n_time, n_energy) = (440, 400)`.

In [ ]:
from scipy.ndimage import gaussian_filter1d

from trspecfit.functions.energy import GLP, LinBack

# --- Ground truth (edit and re-run to see the comparison react) ----------
M_TRUE = 0.85    # GLP mixing: 0 = pure Gaussian, 1 = pure Lorentzian
A_TRUE = 10.0
F_TRUE = 1.0
X0_BASE = 8.0    # peak position at t < 0 and t -> ∞
DELTA_X = 2.0    # kick amplitude at t = 0 (matches expFun A)
TAU = 30.0       # decay time back to X0_BASE
SD_TRUE = 3.0    # instrument-response Gaussian σ (time units)
LIN_M = 0.05
LIN_B = 1.0
NOISE_REL = 0.02

# --- Axes -----------------------------------------------------------------
energy = np.linspace(0.0, 20.0, 400)
time = np.linspace(-10.0, 99.0, 440)

# --- Time-dependent peak position: kicked decay convolved with the IRF ----
# Bare kicked decay: 0 for t < 0, DELTA_X * exp(-t/TAU) for t >= 0.
# m2dB (no IRF) fits this sharp turn-on directly; m2dA convolves it with
# a Gaussian first. We bake the IRF into the truth so the "with IRF" model
# is right and the "no IRF" model is forced to compromise around t = 0.
kick_sharp = np.where(time < 0, 0.0, DELTA_X * np.exp(-time / TAU))
dt = float(time[1] - time[0])
kick_smoothed = gaussian_filter1d(kick_sharp, sigma=SD_TRUE / dt, mode="nearest")
x0_t = X0_BASE + kick_smoothed

# --- Build the clean 2D model: GLP shifted in time + linear background ----
back = LinBack(energy, LIN_M, LIN_B, energy.min(), energy.max(), spectrum=None)
clean = np.stack(
    [GLP(energy, A_TRUE, x0, F_TRUE, M_TRUE) + back for x0 in x0_t]
)

# --- Add noise; NOISE_SIGMA goes into file.set_sigma below ----------------
NOISE_SIGMA = NOISE_REL * clean.max()
rng = np.random.default_rng(42)
data = clean + rng.normal(scale=NOISE_SIGMA, size=clean.shape)

# --- Wire into a Project / File ------------------------------------------
project = trspecfit.Project(path=os.getcwd(), name="model_comparison")
file = trspecfit.File(
    parent_project=project,
    path="synthetic",
    data=data,
    energy=energy,
    time=time,
)
# Register the per-pixel σ on the file once, persistently. Every subsequent
# fit on this file materializes σ-calibrated chi2 / chi2_red into its saved
# slot; compare_models() then shows the canonical "≈ 1 for a good fit"
# reading automatically (no per-call sigma= kwarg). set_sigma() only
# affects *future* fits — slots already in _fit_history keep the σ that
# was materialized at their fit completion. The return value is the
# previous sigma_data (None when unset), available if you want to restore
# it after running a batch of fits with a different σ.
file.set_sigma(NOISE_SIGMA)

print(f"File name:        {file.name}")
print(f"Data shape:       {file.data.shape}")
print(f"True GLP m:       {M_TRUE}   (baseB pins m = 0 → wrong line shape)")
print(f"True IRF σ:       {SD_TRUE}   (m2dB drops the gaussCONV → sharp turn-on)")
print(f"file.sigma_data:  {file.sigma_data:.4f}   "
      f"(noise_type={file.noise_type!r})")
print(f"x0 trajectory:    starts at {x0_t[0]:.2f}, peaks at {x0_t.max():.2f}, "
      f"ends at {x0_t[-1]:.2f}")

## 2. Fit Region and Baseline Window

Same limits for both models so the comparison is on a single fixed grid; the per-slot `observed_sha256` cross-check guards against accidental window drift between fits.

The baseline window is kept tight enough that the IRF-smeared rise can't leak backwards into it. A wider window would contaminate the baseline shape fit — and the contamination grows with `SD_TRUE`, so widen `time_stop` only as much as the chosen IRF allows.

In [ ]:
file.set_fit_limits(energy_limits=[5, 18], time_limits=[-10, 99])
file.define_baseline(time_start=0, time_stop=5, time_type="ind")

## 3. Fit Two Baseline Models

Each `fit_baseline` call appends a `SavedFitSlot` to `Project._fit_history`, so even though `file.model_base` is overwritten by the second fit, both results stay live in memory for the comparison.

Order matters here for a *different* reason: subsequent SbS and 2D fits seed their pinned (non-varying) parameters from `file.model_base.result` — i.e., the **most recent** baseline. We fit `baseB` first and `baseA` last so the GLP shape (the correct one) is what gets pinned downstream.

In [ ]:
# Model B first: Gaussian only (m = 0 pinned)
file.load_model(model_yaml="models_energy.yaml", model_info="baseB")
file.fit_baseline(model_name="baseB", stages=2)

# Model A last: full GLP — leaves model_base.result holding the correct shape
file.load_model(model_yaml="models_energy.yaml", model_info="baseA")
file.fit_baseline(model_name="baseA", stages=2)

print(f"\nSlots in _fit_history: {len(project._fit_history)}")
print("Models seen:", project.results.models(file=file.name))

## 4. `file.compare_models(...)` — Baseline

Returns a `pandas.DataFrame` keyed by `(file, model, fit_type, selection_json)` plus one column per requested metric.

**Stable column meanings.** lmfit's raw `chi2_red = Σ(observed − fit)² / (N − p)` has units of *(data units)²* and only hits the canonical "≈ 1 for a good fit" benchmark when residuals are weighted by σ. Rather than rename the same column from raw to calibrated depending on context, the schema separates them:

- `chi2_red_raw` is **always present** (the lmfit-unweighted diagnostic).
- `chi2_red` is the σ-calibrated value — present **only** when a sigma was set on the file at fit time (we did that in §1 via `file.set_sigma(NOISE_SIGMA)`).
- `sigma_eff` sits between them and shows the per-row σ that was actually applied. Baseline rows get auto-corrected by `√N_avg` (from the slot's `base_t_ind`); SbS, 2D, and spectrum rows use σ verbatim.

The same column name always carries the same kind of value across calls, sessions, and loaded archives. There is **no per-call `sigma=` kwarg** — sigma is persistent state on the File and is materialized into each saved slot at fit time.

`baseB` will land well above 1.0 since a pure Gaussian can't represent the Lorentzian tails.

In [ ]:
file.compare_models(fit_type="baseline")

### Smoke-test residual plot

`FitResults.plot_residuals` is intentionally minimal (index axes, no energy/time labels) — it's a quick visual cross-check, not a publication figure. Build your own from `slot.observed` / `slot.fit` for anything fancier.

In [ ]:
project.results.plot_residuals(
    file=file,
    models=["baseA", "baseB"],
    fit_type="baseline",
);

## 5. Fit Two Slice-by-Slice Models

Pinned (non-varying) parameters seed from `file.model_base.result` — `baseA` at this point, since we fit it last in section 3.

- `sbsA`: only `x0` floats per slice (truth-like — the simulation moves only `x0`).
- `sbsB`: `x0` *and* peak amplitude `A` both float per slice (extra free parameter).

In [ ]:
file.load_model(model_yaml="models_energy.yaml", model_info="sbsA")
file.fit_slice_by_slice(model_name="sbsA", stages=1, try_ci=0)

file.load_model(model_yaml="models_energy.yaml", model_info="sbsB")
file.fit_slice_by_slice(model_name="sbsB", stages=1, try_ci=0)

print("\nSbS slots:")
for slot in project.results.find(file=file.name, fit_type="sbs"):
    # chi2_red_raw is always populated; chi2_red is σ-calibrated.
    n_slices = len(slot.metrics["chi2_red_raw"])
    print(f"  {slot.model_name}: {n_slices} slices")

## 6. SbS Comparison — Aggregation Modes

Per-slice metrics need to be collapsed to compare SbS slots in a single row. `sbs_aggregation` controls how:

- `"median"` (default) — robust scalar via `np.nanmedian`.
- `"mean"`   — `np.nanmean`.
- `"sum"`    — `np.nansum` for additive metrics (`chi2_raw`, `chi2`, `aic`, `bic`). Both reduced χ² flavors (`chi2_red_raw`, `chi2_red`) instead aggregate as `Σ numerator / Σ DoF` so the canonical "≈ 1" reading is preserved. `r2` is still nansum'd; treat it as informational in sum mode (no per-slice SST is stored to compute an aggregate r²).
- `"long"`   — bypass aggregation entirely; emit **one row per slice** with a `slice_index` column. Useful for seeing whether a model wins uniformly across time or only in some regime. `sigma_eff` is a per-fit scalar and is broadcast to every slice row.

In [ ]:
# Default (median) aggregation — one row per slot
file.compare_models(fit_type="sbs")

In [ ]:
# Same comparison, sum-aggregated. AIC/BIC and chi2/chi2_raw become additive
# across slices; chi2_red and chi2_red_raw use Σnumerator / ΣDoF so they stay
# at the canonical "≈ 1 for a good fit" scale instead of growing with N_slices.
file.compare_models(fit_type="sbs", sbs_aggregation="sum")

In [ ]:
# Long form — one row per slice. Filter to a metric of interest and look
# at how the two models track each other across time.
long_df = file.compare_models(fit_type="sbs", sbs_aggregation="long")
print(f"long-form rows: {len(long_df)}")
long_df.head(6)

In [ ]:
# A quick pivot: chi2_red per slice for each model, side by side.
wide = long_df.pivot(index="slice_index", columns="model", values="chi2_red")
wide.describe()

## 7. 2D Comparison — Does the IRF Earn Its Keep?

The truth `x0(t)` is an exponential rise *convolved with a Gaussian IRF*. We fit two competing time-dependence models on the same energy backbone:

- **`m2dA` + `MonoExpPosIRF`** — exponential rise convolved with `gaussCONV`. Matches truth structurally. One extra free parameter (`gaussCONV SD`).
- **`m2dB` + `MonoExpPos`** — bare exponential rise, no IRF. Sharp turn-on at `t = 0`; cannot reproduce the smooth onset.

This is the inverse of the SbS story: there, the extra free parameter didn't earn its keep. Here, it should — the IRF is structurally required and BIC will reward `m2dA` despite the parameter-count penalty.

In [ ]:
# Model A: with IRF (gaussCONV ⊗ expFun) — matches truth
file.load_model(model_yaml="models_energy.yaml", model_info="m2dA")
file.add_time_dependence(
    target_model="m2dA",
    target_parameter="GLP_01_x0",
    dynamics_yaml="models_time.yaml",
    dynamics_model="MonoExpPosIRF",
)
file.fit_2d(model_name="m2dA", stages=2, try_ci=0)

In [ ]:
# Model B: no IRF (sharp expFun rise) — structurally wrong at t ≈ 0
file.load_model(model_yaml="models_energy.yaml", model_info="m2dB")
file.add_time_dependence(
    target_model="m2dB",
    target_parameter="GLP_01_x0",
    dynamics_yaml="models_time.yaml",
    dynamics_model="MonoExpPos",
)
file.fit_2d(model_name="m2dB", stages=2, try_ci=0)

In [ ]:
file.compare_models(fit_type="2d")

## 8. Save → Load → Compare Across Sessions

`file.save_fit(path)` is sugar for `project.save_fits(path, file=file)`. By default it writes a snapshot of `_fit_history` (latest slot per canonical key, `keep_history=True` is deferred to v2). `FitResults.load(path)` reads the archive without needing a `Project`, so this is the persistence path that survives kernel restarts and travels well between machines.

Each saved slot carries its own σ snapshot (`noise_type`, `sigma_source`, `sigma_type`, `sigma_data`, `sigma_eff`), so a loaded archive shows the **same calibrated columns** the live session did — no need to re-`set_sigma` on the recipient side.

**What if you disagree with the saved σ?** The raw column `chi2_red_raw` is always present, so a what-if recalibration is one line of pandas — no rehydrated Project required:

```python
df = loaded.compare_models(file=file.name, fit_type="2d")
df["chi2_red_alt"] = df["chi2_red_raw"] / alt_sigma**2
```

In [ ]:
archive_path = "comparison.fit.h5"
file.save_fit(archive_path, overwrite=True)

loaded = trspecfit.FitResults.load(archive_path)
print(loaded)

In [ ]:
# Same comparison API, but driven from the on-disk archive — no Project required.
# Each slot stores its σ snapshot at fit time, so calibrated columns appear here
# even though we never called set_sigma() on this fresh FitResults.
loaded.compare_models(file=file.name, fit_type="baseline")

In [ ]:
loaded.compare_models(
    file=file.name, fit_type="sbs", sbs_aggregation="long"
).head(6)

## 9. Browse the Loaded Archive

Beyond `compare_models`, `FitResults` exposes a small query API: `files()`, `models()`, `find()` (multi-match), and `get()` (exactly one match, raises otherwise). Slots carry their fingerprint, identity, params DataFrame, observed/fit arrays, and metrics — everything needed to inspect a fit without rehydrating the model graph.

In [ ]:
print("files:  ", loaded.files())
print("models: ", loaded.models())

slot = loaded.get(file=file.name, model="baseA", fit_type="baseline")
print(f"\nbaseA baseline slot:")
print(f"  observed shape: {slot.observed.shape}")
print(f"  fit shape:      {slot.fit.shape}")
print(f"  metrics:        {slot.metrics}")
print(f"  params rows:    {len(slot.params)}")
slot.params

## Tips

- `Project.results` is a property — every access returns a fresh `FitResults` snapshot. Capture it (`r = project.results`) to freeze the slot list against subsequent fits, or re-access for the latest view.
- `file.compare_models(...)` is sugar for `project.results.compare_models(file=file, ...)`. The implementation lives entirely on `FitResults`, so loaded archives use the exact same code path.
- **`File.set_sigma()` is the only sigma entry point.** `compare_models()` has no `sigma=` kwarg by design — column names are stable across calls, sessions, and loaded archives. `set_sigma()` only affects **future** fits on the file; existing slots in `Project._fit_history` (and any saved archive) keep the σ snapshot that was materialized at their fit completion. For what-if recalibration of *existing* results, divide the always-present `chi2_red_raw` column by `alt_sigma**2` directly:

  ```python
  df = file.compare_models(fit_type="2d")
  df["chi2_red_alt"] = df["chi2_red_raw"] / alt_sigma**2
  ```

- The archive is **slot-scoped on overwrite**: re-saving with the same `(file, model, fit_type, selection)` raises unless `overwrite=True`. Pass a fresh path to start clean.
- For human-readable artifacts (CSV/PNG tree instead of HDF5), use `file.export_fit(...)` or `project.export_fits(...)` — same filter pipeline, different sink.